In [ ]:
Sys.setenv(LANGUAGE = "en")

library(readxl)
library(dplyr)
library(survival)
library(survminer)
library(tidyr)
library(ggplot2)
library(extrafont)
library(svglite)
library(ComplexHeatmap)

library(ggsurvfit)
library(stringr)


In [ ]:
# imports from external file
imports <- new.env()
source("survival-data-imports.R", local = imports)
plotting <- new.env()
source("survival-plots.R",local=plotting)
source("../../src/plotting.R",local=plotting)

sessionInfo()

In [ ]:
dir.create('hgg', showWarnings = FALSE)

In [ ]:
import_hgg_data = function(path = "../../manuscript/collaborators/Rishaan/pHGG/pHGG_Survival_Data_Merged.xlsx"){
    # 2026/3 Changelog: wrap in function, censor at 5y, drop HGNET, cleanup (drop unused transformations, drop_na), add ecDNA_status and amplified
    df <- read_excel(path)
    df <- df %>% 
        rename_with(tolower) %>%
        drop_na(os_months,os_status) %>%
        filter(cancer_subclass != 'HGNET') %>%
        mutate(
            os_status = tolower(os_status),
            os_event = case_when(
                os_status %in% c("deceased", "dead") ~ 1,
                os_status %in% c("alive", "living") ~ 0,
                TRUE ~ NA_real_),
            os_event = if_else(os_months <= 60, os_event, 0),
            os_months = if_else(os_months < 60, os_months, 60),
            amplicon_class = case_when(
              amplicon_class %in% c("Chromosomal", "intrachromosomal") ~ "chromosomal",
              TRUE ~ amplicon_class
            ),
            ecDNA_status = ifelse(amplicon_class == "ecDNA","ecDNA+","ecDNA-"),
            amplified = ifelse(amplicon_class %in% c('ecDNA','chromosomal'),"amp+","amp-")
        ) %>% 
        select(-os_status)
}

factorize = function(df){
    # convert string columns to factors.
    transformations = list(
        cancer_subclass = . %>% factor(),
        cancer_subclass2 = . %>% factor(),
        H3G34 = . %>% factor(),
        H3K27 = . %>% factor(),
        TP53 = . %>% factor() %>% relevel(ref="WT"),
        IDH = . %>% factor(),
        hgg_subtype = . %>% factor() %>% relevel(ref="H3WT"),
        amplicon_class = . %>% factor() %>% relevel(ref="no amplification"),
        ecDNA_status = . %>% factor() %>% relevel(ref="ecDNA-"),
        chromosomal_amp = . %>% factor() %>% relevel(ref="amp-"),
        amplified = . %>% factor() %>% relevel(ref="amp-")
    )
    df <- purrr::reduce(names(transformations), function(df, col) {
        if (!col %in% names(df)) return(df)
        mutate(df, across(all_of(col), transformations[[col]]))
    }, .init = df)
    return(df)
}
# Usage:
df = import_hgg_data() %>% factorize
print(table(df$amplicon_class))
print(table(df$os_event))

In [ ]:
## Reusable functions for annotating subgroups, mutations, and interaction terms.
## Note that ggforest can't plot interaction terms generated with interaction(), hence the add_h3k27_interactions function to explicitly add those to the data.
## 2026/3 Changelog: 
# Other -> IHG, IDH, NOS
# Bugfix mutation detection logic - NOS classes shouldn't be set to NA, and one of them is TP53
# delete redundant droplevels, KM plots, H3WT variable
# rename combined_cancer_subclass -> hgg_subtype

#Combining similar HGG subclasses
aggregate_hgg_mutations = function(df){
    h3k27_classes <- c("DMG_H3K27", "DMG_H3K27M", "HGG_H3K27", "DMG_H3K27_TP53")
    h3wt_classes  <- c("HGG_H3WT", "HGG_H3WT_TP53")
    h3g34_classes <- c("DHG_H3G34", "DHG_H3G34_TP53", "HGG_H3G34")
    collapse_to_other <- c("IHG_ALK", "IHG_NRTK", "IHG_ROS1_TP53", "IHGNOS")
    df = df %>%
        mutate(
            hgg_subtype = case_when(
                cancer_subclass %in% collapse_to_other ~ "IHG",
                cancer_subclass %in% h3k27_classes ~ "H3K27-altered",
                cancer_subclass %in% h3wt_classes  ~ "H3WT", ###
                cancer_subclass %in% h3g34_classes  ~ "H3G34-altered",
                cancer_subclass == 'HGG_IDH_TP53' ~ 'IDH',
                cancer_subclass == 'HGG_NOS' ~ 'NOS',
                TRUE ~ cancer_subclass #stop('unknown cancer_subclass')
            ),
            TP53 = ifelse(grepl("TP53", cancer_subclass, ignore.case = TRUE),"mut","WT"),
            H3K27 = ifelse(grepl("H3K27", cancer_subclass, ignore.case=TRUE),1,0),
            H3G34 = ifelse(grepl("H3G34", cancer_subclass, ignore.case=TRUE),1,0),
            IDH = ifelse(grepl("IDH", cancer_subclass, ignore.case=TRUE),1,0),
        ) %>%
        factorize() %>% 
        droplevels()
}

add_h3k27_interactions = function(df){
    # Create a interaction terms for h3k27 x amplification; and, because ggforest can't plot interactions, manually
    # add these as independent variables
    h3k27_labels = c("1"="H3K27mut","0"="H3K27wt")
    ampclass_labels = c("no amplification"="nfa","chromosomal"="chr","ecDNA"="ecDNA")
    df %>% mutate(
        h3k27_x_amp = ifelse(H3K27==1 & amplified == 'amp+',1,0),
        h3k27_x_ecDNA = ifelse(H3K27==1 & ecDNA_status == 'ecDNA+',1,0),
        h3k27_x_chr = ifelse(H3K27==1 & amplicon_class == 'chromosomal',1,0),
        ecDNA_H3K27 = interaction(ecDNA_status, fct_relabel(H3K27,~ h3k27_labels[.x]), drop = TRUE),
        ampclass_H3K27 = interaction(
            fct_relabel(amplicon_class,~ ampclass_labels[.x]),
            fct_relabel(H3K27,~ h3k27_labels[.x]), 
            drop = TRUE)
    )
}

# Usage:
df <- import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>% add_h3k27_interactions
subclass_counts <- df %>% count(hgg_subtype)
print(subclass_counts)
df %>% head

### SF8a KM stratified by H3K27 mutation status
and
### Not included: KM stratified by amplicon class

In [ ]:
# KM stratified by H3K27 mutation status
h3k27_labels = c("1"="H3K27mut","0"="H3K27wt")
df = import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>%
    mutate(
        H3K27 = fct_relabel(H3K27,~ h3k27_labels[.x])
    )

formula = Surv(os_months, os_event) ~ H3K27
km_fit <- survfit(formula, data = df)

options(repr.plot.width = 6, repr.plot.height = 6)
colors = setNames(c('#2E8B57','#FF4500'),str_replace(names(km_fit$strata), "^.*=", ""))
plotting$km_plot(km_fit, colors)
plotting$save_ggplot("km_hgg_h3k27mut_5year",path='hgg')
#km_plot(km_fit,"hgg/pHGG_Survival_h3k27")

# Print log-rank test p-values
logrank <- survdiff(formula, data = df)
logrank

In [ ]:
# KM stratified by subgroup
df = import_hgg_data() %>% factorize %>% aggregate_hgg_mutations
formula = Surv(os_months, os_event) ~ hgg_subtype
km_fit <- survfit(formula, data = df)

options(repr.plot.width = 6, repr.plot.height = 6)
subtype_palette <- c(
  "H3WT"          = "#0072B2",  # blue
  "H3G34-altered" = "#009E73",  # green
  "H3K27-altered" = "#D55E00",  # vermillion
  "IDH"           = "#56B4E9",  # sky blue
  "IHG"           = "#E69F00",  # orange
  "NOS"           = "#CC79A7"   # pink
)
plotting$km_plot(km_fit,subtype_palette)
plotting$save_ggplot("km_hgg_subgroups_5year",path='hgg')

# Print log-rank test p-values
logrank <- pairwise_survdiff(formula, data = df, p.adjust.method = "BH") 
print(logrank)


In [ ]:
# KM stratified by amplicon class
df = import_hgg_data() %>% factorize
formula = Surv(os_months, os_event) ~ amplicon_class
km_fit <- survfit(formula, data = df)

options(repr.plot.width = 6, repr.plot.height = 6)
plotting$km_plot(km_fit)
plotting$save_ggplot("km_hgg_5year",path='hgg')
#km_plot(km_fit,"hgg/KM_Survival_AmpliconClass")

# Print log-rank test p-values
logrank <- pairwise_survdiff(formula, data = df, p.adjust.method = "BH")
logrank_df <- as.data.frame(as.table(as.matrix(logrank$p.value)))
p_ecDNA <- logrank_df %>% filter(Var1 == "ecDNA" | Var2 == "ecDNA") %>% mutate(comparison = paste(Var1, "vs", Var2), p_value = signif(Freq, 3)) %>%
  select(comparison, p_value)
print(logrank)
pval_labels <- paste0(p_ecDNA$comparison, " (p=", p_ecDNA$p_value, ")")
cat(paste0(p_ecDNA$comparison, ": ", p_ecDNA$p_value, collapse = "\n"))

In [ ]:
df <- import_hgg_data() %>% factorize %>% aggregate_hgg_mutations
df %>% head

### SF8b Cox model on amplification status, ecDNA, and molecular subgroups defined by prognostic mutations
Does ecDNA  affect survival independent of molecular subgroup?  
--> No

2026/03 Changelog:  
- add TP53 term  
- chomosomal_amp -> amplified  

In [ ]:
cox_model <- coxph(
  Surv(os_months, os_event) ~ ecDNA_status + amplified + hgg_subtype + TP53,
  data = df
)
#plotting$cox_plot(cox_model)
#plotting$save_ggplot(filename="hgg/pHGG_Cox_subgroups_p53_alt",width=6,height=6)
p <- plotting$forest_coxph(cox_model)
plotting$show_forestploter(p)
plotting$save_forestploter(p,'hgg/pHGG_Cox_subgroups_p53_alt.png')
plotting$save_forestploter(p,'hgg/pHGG_Cox_subgroups_p53_alt.svg')

# Alternately, using amplicon_class. Same model, different reference levels.
#cox_model <- coxph(
#  Surv(os_months, os_event) ~ amplicon_class + hgg_subtype + TP53,
#  data = df
#)

#cox_plot(cox_model,outfile="hgg/pHGG_Cox_subgroups_p53",width=6,height=4.5)

In [ ]:
# SF8b proportional-hazards check. The violation is in hgg_subtype (chisq p ~ 0.02), the
# adjustment covariate, not in ecDNA_status / amplified (the exposures, both p > 0.4). So the
# "ecDNA/amplification not independently prognostic" conclusion stands; the subtype HRs this
# panel reports are time-averaged summaries of a non-proportional (front-loaded H3K27) effect.
# Dropping that assumption means stratifying the baseline on subtype (as in the stratified SF8d
# below), which forgoes the very subtype HRs this panel exists to show -- hence not done here.
cox.zph(cox_model)

### SF8c
KM stratified by H3K27mut and ecDNA. Chromosomal or extrachromosomal amplification nominally but non-significantly associated with poorer 5-year survival

In [ ]:
# KM stratified by H3K27 status and amplicon class.
formula = Surv(os_months, os_event) ~ ampclass_H3K27
df = import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>% add_h3k27_interactions
fit <- survfit(formula, data = df)

logrank <- pairwise_survdiff(formula, data = df, p.adjust.method = "BH") 
# get colors with ggplot_build(plt$plot)$data[[1]] %>% group_by(group) %>% slice(1) %>% pull(colour)
colors = setNames(c('#F8766D','#B79F00','#00BA38','#00BFC4','#619CFF','#F564E3'),
                  str_replace(names(fit$strata), "^.*=", ""))
#km_plot(fit,"hgg/pHGG_Survival_ecDNA_x_h3k27",colors=colors)
options(repr.plot.width = 6, repr.plot.height = 6)
plotting$km_plot(fit,palette=colors)
plotting$save_ggplot("km_hgg_5year",path='hgg')
print(logrank)

In [ ]:
# Confusion matrix for p-values, since pairwise across six comparisons is a lot
asdf = Heatmap(
    logrank$p.value,
    cluster_rows = FALSE,
    cluster_columns = FALSE,
    cell_fun = function(j, i, x, y, w, h, col) { # add text to each grid
        grid.text(logrank$p.value[i, j] %>% round(4), x, y)},
    na_col = "white",
    row_names_side = "left",
    show_heatmap_legend = FALSE,
    col = colorRamp2(c(0,0.05, 1), c("steelblue", "skyblue","white")),
)

asdf
svglite("hgg/pHGG_pvals_ecDNA_x_h3k27.svg", width = 5, height = 5)
draw(asdf)
dev.off()

### SF8d. Interaction between H3K27mut and ecDNA.
Does ecDNA effect depend on mutation status (is ecDNA worse in H3K27 tumors than in H3WT tumors)?

*interpretation*: ecDNA is prognostic in gliomas lacking H3K27 alteration, but its effect is neutralized 
in the context of H3K27-altered disease, where baseline risk is already extreme.

In [ ]:
df = import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>% add_h3k27_interactions

cox_interaction <- coxph(
    Surv(os_months, os_event) ~ ecDNA_status + amplified + hgg_subtype + TP53 + h3k27_x_ecDNA + h3k27_x_amp,
    data = df
)
summary(cox_interaction)

#ggforest(cox_interaction, data = as.data.frame(df))
p <- plotting$forest_coxph(cox_interaction)
plotting$show_forestploter(p)
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_alt.png')
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_alt.svg')

In [ ]:
# SF8d proportional-hazards check (covariate model). hgg_subtype (proportional here) violates
# PH; the interaction terms of interest do not. Addressed by the stratified model below.
cox.zph(cox_interaction)

In [ ]:
# SF8d, stratified. Stratify the baseline hazard on molecular subtype (absorbing the
# non-proportional subtype effect flagged by cox.zph, and the noisy rare-subtype coefficients)
# while retaining the H3K27 x ecDNA / x amplification interactions. TP53 stays a covariate:
# it cross-cuts the (mutually exclusive) subtype partition. Estimates are materially unchanged
# vs the covariate model (ecDNA in H3K27-wt HR ~1.77; interaction HR ~0.68), but cox.zph is clean.
df <- import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>% add_h3k27_interactions

cox_strat <- coxph(
    Surv(os_months, os_event) ~ ecDNA_status + amplified + TP53 + h3k27_x_ecDNA + h3k27_x_amp + strata(hgg_subtype),
    data = df
)
summary(cox_strat)
cox.zph(cox_strat)

p <- plotting$forest_with_strata(cox_strat, strata_label = "Molecular subtype")
plotting$show_forestploter(p)
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_stratified.png')
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_stratified.svg')

## Archive
Plots not used in the latest analysis.

In [ ]:
library(gridExtra)
library(forcats)
library(circlize)
library(grid)
library(patchwork)

In [ ]:
## ecDNA+/- KM curves.
## now nominally nonsignificant (p=0.065).

df <- import_hgg_data() %>% factorize
formula = Surv(os_months, os_event) ~ ecDNA_status
km_fit <- survfit(formula, data = df)
logrank <- pairwise_survdiff(formula, data = df, p.adjust.method = "BH")
logrank_df <- as.data.frame(as.table(as.matrix(logrank$p.value)))
p_ecDNA <- logrank_df %>%
  filter((Var1 == "ecDNA+" & Var2 == "ecDNA-") | (Var1 == "ecDNA-" & Var2 == "ecDNA+")) %>%
  mutate(
    comparison = paste(Var1, "vs", Var2),
    p_value = signif(Freq, 3)
  ) %>%
  select(comparison, p_value)
pval_num <- p_ecDNA$p_value  
summary_table <- tableGrob(
  p_ecDNA,
  rows = NULL,
  theme = ttheme_minimal(base_size = 12)
)
print(logrank)

plotting$km_plot(km_fit)
#plotting$save_ggplot("hgg/pHGG_Survival_ecDNA_vs_ecDNAminus",out='hgg')

In [ ]:
#Cox model for H3K27 interaction with amplicon_class
# Note: cox_plot, forest_coxph don't plot interactions
cox_interaction <- coxph(
Surv(os_months, os_event) ~ amplicon_class * H3K27 + H3G34 + TP53,
data = df
)
summary(cox_interaction)
#cox_plot(cox_interaction)
#plotting$save_ggplot(filefile="hgg/pHGG_Cox_interaction_term",width=6,height=5)
p <- plotting$forest_coxph(cox_interaction)
plotting$show_forestploter(p)

hr_with_ci <- function(fit, terms) {
  # Function to calculate hazards relative to levels other than reference
  coefs <- coef(fit)
  v <- vcov(fit)
  
  log_hr <- sum(coefs[terms])
  se <- sqrt(sum(v[terms, terms]))
  
  list(
    hr  = exp(log_hr),
    ci  = exp(log_hr + c(-1, 1) * 1.96 * se),
    p   = 2 * pnorm(-abs(log_hr / se))
  )
}
# hazard for ecDNA given H3K27M-altered
hr_with_ci(cox_interaction,c('amplicon_classecDNA','amplicon_classecDNA:H3K271'))

In [ ]:
# Alternative 8d
df = import_hgg_data() %>% factorize %>% aggregate_hgg_mutations %>% add_h3k27_interactions

cox_interaction <- coxph(
    Surv(os_months, os_event) ~ amplicon_class + hgg_subtype + TP53 + h3k27_x_ecDNA + h3k27_x_amp,
    data = df
)
summary(cox_interaction)

#ggforest(cox_interaction, data = as.data.frame(df))
p <- plotting$forest_coxph(cox_interaction)
plotting$show_forestploter(p)
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_term.png')
plotting$save_forestploter(p,'hgg/pHGG_Cox_interaction_term.svg')

In [ ]:
# KM, ecDNA x H3K27
fit <- survfit(Surv(os_months, os_event) ~ ecDNA_H3K27, data = df)
ggsurvplot(
  fit,
  data = df,
  pval = TRUE,
  risk.table = TRUE,
  legend.title = "ecDNA × H3 status",
  legend.labs = levels(df$ecDNA_H3K27)
)

# KM, amp_class x H3K27
fit <- survfit(Surv(os_months, os_event) ~ ampclass_H3K27, data = df)
ggsurvplot(
  fit,
  data = df,
  pval = TRUE,
  risk.table = TRUE,
  legend.title = "amp. class ×\nH3 status",
  legend.labs = levels(df$ampclass_H3K27)
)

In [ ]:
cox_unadj <- coxph(Surv(os_months, os_event) ~ ecDNA_status, data = df)

cox_adj <- coxph(
  Surv(os_months, os_event) ~ ecDNA_status + H3K27 + H3G34 + TP53,
  data = df
)

cox_interaction <- coxph(
  Surv(os_months, os_event) ~ ecDNA_status * H3K27 + H3G34 + TP53,
  data = df
)

#extract ecDNA effect from each model
extract_ecDNA <- function(model, label) {
  tidy(model, exponentiate = TRUE, conf.int = TRUE) %>%
    filter(term == "ecDNA_statusecDNA+") %>%
    mutate(model = label)
}

forest_multi <- bind_rows(
  extract_ecDNA(cox_unadj, "Unadjusted"),
  extract_ecDNA(cox_adj, "Adjusted for mutations"),
  extract_ecDNA(cox_interaction, "With H3K27 interaction")
)

#Forest plot
p_forest_multi <- ggplot(
  forest_multi,
  aes(x = estimate, y = model)
) +
  geom_vline(xintercept = 1, linetype = "dashed", color = "grey40") +
  geom_point(size = 4) +
  geom_errorbarh(
    aes(xmin = conf.low, xmax = conf.high),
    height = 0.25,
    linewidth = 1
  ) +
  scale_x_log10() +
  labs(
    title = "Effect of ecDNA on Overall Survival Across Models",
    x = "Hazard Ratio (log scale)",
    y = NULL
  ) +
  theme_minimal(base_size = 28) +
  theme(
    plot.title = element_text(face = "bold", size = 32),
    axis.text = element_text(size = 24)
  )

ggsave(
  "hgg/Forest_ecDNA_ModelComparison.png",
  p_forest_multi,
  width = 20.33,
  height = 10,
  dpi = 300
)

forest_df
forest_multi